# 폰트설정
- 아래 코드 선 실행
- seaborn, matplotlib 라이브러리 활용 시, 한글폰트 깨짐 방지
- 런타임 > 세션 다시 시작
  + 아래 코드 생략, 다음 코드 실행
- Google Colab

In [39]:
# ── 환경 설정(선택) ─────────────────────────────────────────────────────
# Colab·리눅스 등에서 폰트 설치 셀 — 로컬 Windows는 생략 가능

!sudo apt-get install -y -qq fonts-nanum
!fc-cache -fv
!rm ~/.cache/matplotlib -rf


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share

In [40]:
# ── Matplotlib ────────────────────────────────────────────────────
# 그래프·축·스타일

import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

def setup_matplotlib_korean_font() -> str | None:
    """Windows / Mac / 기타 순으로 한글 폰트를 try-except로 적용."""
    try:
        plt.rcParams["axes.unicode_minus"] = False
    except Exception:
        pass

    system = platform.system()

    if system == "Windows":
        candidates = ["Malgun Gothic", "맑은 고딕"]
    elif system == "Darwin":
        candidates = ["Apple SD Gothic Neo", "AppleGothic"]
    else:
        candidates = ["NanumGothic", "Noto Sans CJK KR", "Malgun Gothic"]

    for family in candidates:
        try:
            prop = font_manager.FontProperties(family=family)
            path = font_manager.findfont(prop, fallback_to_default=False)
            if path and "dejavu" not in path.lower():
                plt.rcParams["font.family"] = family
                print(f"[한글 폰트] 적용: {family}")
                return family
        except Exception as exc:
            print(f"[한글 폰트] 후보 실패 ({family}): {exc}")
            continue

    # 검증 없이 플랫폼 기본 이름만 지정 (실제 글리프는 런타임에 확인)
    try:
        if system == "Windows":
            plt.rcParams["font.family"] = "Malgun Gothic"
            print("[한글 폰트] 폴백(Windows): Malgun Gothic")
            return "Malgun Gothic"
        if system == "Darwin":
            plt.rcParams["font.family"] = "Apple SD Gothic Neo"
            print("[한글 폰트] 폴백(macOS): Apple SD Gothic Neo")
            return "Apple SD Gothic Neo"
    except Exception as exc:
        print(f"[한글 폰트] 폴백 실패: {exc}")

    print("[한글 폰트] 시스템 기본만 사용 — 한글이 깨질 수 있습니다.")
    return None

setup_matplotlib_korean_font()


[한글 폰트] 후보 실패 (NanumGothic): Failed to find font NanumGothic:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled
[한글 폰트] 후보 실패 (Noto Sans CJK KR): Failed to find font Noto Sans CJK KR:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled
[한글 폰트] 후보 실패 (Malgun Gothic): Failed to find font Malgun Gothic:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled
[한글 폰트] 시스템 기본만 사용 — 한글이 깨질 수 있습니다.


In [41]:
# 구글 드라이브와 연결
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Part 2. 정형 데이터 Pandas 튜토리얼 70제 (`dataset/정형`)

`Pandas예제1`~`7`의 학습 흐름(불러오기, `info`/`describe`, `loc`/`iloc`, 조건 필터, 정렬, `groupby`, 결측·중복, 파생, 문자열, 날짜, `merge`, NumPy)을 **아래 세 CSV**에 적용합니다.

| 데이터 | 파일(예시 이름) |
|--------|-----------------|
| 서버 점검 | `서버유지보수_보정데이터(0512).csv` |
| 군수품 청구 | `군수품자동청구_보정데이터(0512).csv` |
| 장병 복지시설 | `장병복지시설용_보완데이터(0512).csv` |

- 경로: 노트북을 `소집교육/0423`에서 열면 `../dataset/정형/`, 프로젝트 루트에서 열면 `소집교육/dataset/정형/`을 자동 탐색합니다.
- 파일명이 유니코드(NFC/NFD)로 달라도 **컬럼 시그니처**(`log_id`, `supply_id`, `soldier_id`)로 구분합니다.

**문제 구성**: `#` 큰 주제 → `##` 세부 주제 → `### 문제 N.` (목표·설명·실습·힌트) → 코드 셀.



## 0. 정형 데이터 로드

**아래 셀을 Part 2 시작 전에 한 번 실행**하세요. `df_srv`, `df_sup`, `df_wel`이 생성됩니다.


In [42]:
import pandas as pd

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/'

# 서버
dr_srv = pd.read_csv(DATA_PATH + "서버데이터.csv")

# 군수품
dr_sup = pd.read_csv(DATA_PATH + "군수품데이터.csv")

# 복지
df_wel = pd.read_csv(DATA_PATH + "장병데이터.csv")
df_wel

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
0,S20240700001,17사단,전라권,36.517251,127.006332,이병,노래방,0,2,6,153.9,격리형,1,2024-09-13,-53.9,1.65,-1.160000,2.15,2.160000
1,S20240700002,14사단,전라권,36.227541,127.840048,일병,독서실,0,2,6,343.1,침상형,1,2024-11-05,-243.1,-3.08,0.816759,-2.58,-2.880000
2,S20240700003,23사단,전라권,37.038748,127.239173,이병,체력단련실,1,0,5,385.8,격리형,1,2024-09-10,-285.8,-4.65,0.898536,-4.15,-3.571377
3,S20240700004,25사단,수도권,38.416642,128.533748,이병,독서실,0,2,3,308.3,침상형,3,2024-12-02,-208.3,-3.71,0.476348,-3.71,-2.770103
4,S20240700005,21사단,수도권,37.097390,127.730811,일병,노래방,1,4,3,121.3,침상형,2,2024-08-09,-21.3,0.97,0.770000,0.97,1.230000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10995,S20240710996,14사단,강원권,37.501527,126.710742,병장,독서실,0,3,6,196.3,격리형,3,2024-11-23,-96.3,0.59,1.970000,1.09,1.030000
10996,S20240710997,18사단,수도권,37.789255,128.201543,병장,체력단련실,1,4,2,447.2,침상형,2,2024-11-17,-347.2,-7.68,0.582506,-7.68,-8.060000
10997,S20240710998,19사단,충청권,36.031731,127.026663,병장,PX,1,5,3,138.1,격리형,2,2024-11-21,-38.1,0.55,1.220000,0.55,0.780000
10998,S20240710999,13사단,수도권,35.806840,127.018945,병장,체력단련실,1,4,7,412.6,격리형,2,2024-07-04,-312.6,-4.32,0.800089,-3.32,-4.140000


# 1. 데이터 읽기와 구조 이해



## 1.1 크기·타입·스키마



### 문제 1.
- **목표**: 데이터프레임의 행·열 규모를 파악한다.
- **설명**: `shape` 속성은 `(행 개수, 열 개수)` 형태의 튜플을 반환합니다. 대용량 여부, 다른 테이블과 행 수 비교, 메모리 추정의 출발점이 됩니다.
- **실습**: `df_srv`의 행·열 개수(`shape`)를 출력하세요.
- **힌트**: `df_srv.shape[0]`은 행, `shape[1]`은 열입니다.


In [43]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
dr_srv.shape

(11000, 15)

In [44]:
dr_srv.shape[0] # 행의 개수 출력

11000

In [45]:
dr_srv.shape[1] # 열의 개수 출력

15

### 문제 2.
- **목표**: 열 이름, dtypes, 비결측 개수 등 스키마를 한 번에 점검한다.
- **설명**: `info()`는 각 열의 이름·dtype·비어 있지 않은 값 개수를 보여 줍니다. `object`는 주로 문자열이며, 날짜가 문자열로 들어온 경우 나중에 `to_datetime`이 필요할 수 있습니다.
- **실습**: `df_sup`에 대해 `info()`를 호출해 dtypes와 결측을 확인하세요.
- **힌트**: 맨 아래 `memory usage`로 대략적인 메모리도 확인할 수 있습니다.


- pandas : 데이터 가공할 때 필요한 라이브러리
- SQL : 데이터 가공
- AI한테 코드 <-> 쿼리 변환하는 거 물어보기

In [46]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
dr_sup.info() # DESC dr_sup

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11000 entries, 0 to 10999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   supply_id            11000 non-null  object 
 1   unit_code            11000 non-null  object 
 2   item_code            11000 non-null  object 
 3   item_name            11000 non-null  object 
 4   category             11000 non-null  object 
 5   stock_level          11000 non-null  float64
 6   avg_daily_usage      11000 non-null  float64
 7   weekend_usage_ratio  11000 non-null  float64
 8   delivery_lead_time   11000 non-null  int64  
 9   requested_quantity   11000 non-null  int64  
 10  reorder_threshold    11000 non-null  int64  
 11  auto_reorder_flag    11000 non-null  int64  
 12  request_date         11000 non-null  object 
 13  is_weekend           11000 non-null  int64  
 14  is_weekend_check     11000 non-null  bool   
dtypes: bool(1), float64(3), int64(5), ob

### 문제 3.
- **목표**: 수치형 열의 분포(평균, 사분위, 최소·최대)를 요약한다.
- **설명**: `describe()`는 기본적으로 수치형 열만 대상으로 합니다. 복지 데이터의 만족도·거리 등 스케일이 다른 열을 비교할 때 유용합니다.
- **실습**: `df_wel`의 수치형 열만 `describe()`로 요약하세요.
- **힌트**: 범주형 요약은 `describe(include='object')`로 별도 확인할 수 있습니다.


In [47]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
df_wel.describe()

,lat,lon,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,mental_wellbeing_score,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
count,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000,11000.000000
mean,37.005726,128.014970,0.501545,2.460273,3.417273,300.390618,2.127182,-200.390618,-3.301126,1.627614,-2.987308,-2.927116
std,0.865462,0.859235,0.500020,1.268795,1.553563,115.049806,1.813845,115.049806,2.985071,2.360851,3.078306,3.145378
min,35.500157,126.501745,0.000000,0.000000,1.000000,100.000000,1.000000,-399.900000,-9.500000,-3.160000,-9.500000,-10.060000
25%,36.252903,127.276596,0.000000,2.000000,2.000000,201.275000,1.000000,-299.925000,-5.760000,0.450000,-5.480000,-5.534524
50%,37.000653,128.015438,1.000000,2.000000,3.000000,299.450000,2.000000,-199.450000,-3.300000,0.980414,-2.990000,-2.838181
75%,37.759142,128.762646,1.000000,3.000000,5.000000,399.925000,2.000000,-101.275000,-0.840000,1.450696,-0.500000,-0.285340
max,38.499901,129.499944,1.000000,7.000000,7.000000,499.900000,10.000000,0.000000,3.460000,17.302844,4.460000,4.981719


In [48]:
# 범주형 변수
df_wel.describe(include = 'object')

,soldier_id,unit_code,region,rank,facility_type,has_private_space,date
count,11000,11000,11000,11000,11000,11000,11000
unique,11000,15,5,4,5,2,184
top,S20240711000,23사단,충청권,상병,독서실,침상형,2024-07-15
freq,1,774,2219,2818,2324,5573,92


### 문제 4.
- **목표**: 분석·코딩 시 참조할 열 이름 목록을 얻는다.
- **설명**: `columns`는 `Index` 객체이며 `.tolist()`로 파이썬 리스트로 바꿀 수 있습니다. 오타 방지와 동적 컬럼 선택에 쓰입니다.
- **실습**: `df_srv.columns.tolist()`로 컬럼 이름 리스트를 보세요.
- **힌트**: 첫 열은 `df.columns[0]`처럼 인덱싱할 수 있습니다.


In [49]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
dr_srv.columns.tolist() # 컬럼명을 리스트 자료형으로 변환

['log_id',
 'server_id',
 'location',
 'check_date',
 'check_type',
 'issue_detected',
 'issue_category',
 'cpu_usage',
 'memory_usage',
 'disk_usage',
 'temp_abnormal_flag',
 'uptime_days',
 'auto_alert_triggered',
 'fix_required',
 'fix_duration_hours']

### 문제 5.
- **목표**: 열 단위로 dtype을 확인한다.
- **설명**: `dtypes`는 Series로, 열 이름 → dtype 매핑입니다. 병합·연산 전에 `int`와 `object` 혼동을 줄입니다.
- **실습**: `df_sup.dtypes`를 출력하세요.
- **힌트**: `df['col'].dtype`으로 한 열만 확인할 수도 있습니다.


In [50]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_sup.dtypes

,0
supply_id,object
unit_code,object
item_code,object
item_name,object
category,object
stock_level,float64
avg_daily_usage,float64
weekend_usage_ratio,float64
delivery_lead_time,int64
requested_quantity,int64


## 1.2 미리보기·표본·메모리



### 문제 6.
- **목표**: 앞쪽 행 몇 개로 값의 형태와 단위를 육안 확인한다.
- **설명**: `head(n)`은 상위 n행입니다. 실제 업무에서는 샘플과 함께 도메인(부대 코드, 품목명 등)이 기대와 맞는지 봅니다.
- **실습**: `df_wel.head(7)`로 앞 7행을 보세요.
- **힌트**: 기본값은 `head()`만 쓰면 5행입니다.


In [51]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

df_wel.head(1)  # SELECT * FROM df_wel LIMIT 1

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
0,S20240700001,17사단,전라권,36.517251,127.006332,이병,노래방,0,2,6,153.9,격리형,1,2024-09-13,-53.9,1.65,-1.16,2.15,2.16


### 문제 7.
- **목표**: 데이터 끝부분(최근 적재 행 등)을 확인한다.
- **설명**: `tail(n)`은 하위 n행입니다. 시계열이 정렬되어 있다면 최근 기록이 아래에 모일 수 있습니다.
- **실습**: `df_srv.tail(3)`로 끝 3행을 보세요.
- **힌트**: 정렬 없이 tail만으로 ‘최근’을 단정하긴 어렵습니다.


In [52]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.tail(2)

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
10998,CHK010999,SRV0199,12사단,2024-12-18 06:00:00,월간,0,없음,13.9,26.2,20.6,0,119,1,0,2.38
10999,CHK011000,SRV0200,25사단,2024-08-12 01:00:00,월간,0,전력,23.5,17.3,39.4,0,181,1,0,1.92


### 문제 8.
- **목표**: 데이터프레임이 차지하는 메모리를 대략 추정한다.
- **설명**: `memory_usage(deep=True)`는 객체 열의 내용까지 더 정밀하게 잡습니다. 합치면 전체 규모의 감을 잡을 수 있습니다.
- **실습**: `df_sup.memory_usage(deep=True).sum()`으로 대략적 메모리를 구하세요.
- **힌트**: 단위는 바이트이므로 1e6으로 나누면 MB 근사가 됩니다.


In [53]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

# 4.6MB 사용 중
dr_sup.memory_usage(deep = True).sum() / 1e6

np.float64(4.600533)

### 문제 9.
- **목표**: 무작위 표본으로 편향된 앞부분만 보는 것을 완화한다.
- **설명**: `sample(n, random_state=)`는 재현 가능한 난수 표본입니다. EDA에서 무작위 몇 행을 보는 습관이 좋습니다.
- **실습**: `df_wel.sample(5, random_state=42)`로 무작위 5행을 보세요.
- **힌트**: `random_state`를 고정해야 노트북 결과가 매번 같습니다.


In [54]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

df_wel.sample(5, random_state = 42)

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
107,S20240700108,18사단,충청권,37.124050,127.579636,상병,독서실,0,3,2,243.8,침상형,2,2024-07-12,-143.8,-2.60,0.042420,-2.60,-1.808633
5484,S20240705485,19사단,충청권,36.781475,127.713932,병장,PX,1,3,3,382.2,침상형,1,2024-10-27,-282.2,-5.56,0.572780,-5.56,-5.730000
6998,S20240706999,20사단,강원권,37.352537,128.669444,병장,체력단련실,1,3,4,223.0,격리형,1,2024-08-24,-123.0,-1.08,1.880000,-0.08,-0.880000
3984,S20240703985,15사단,영남권,35.649815,127.583513,일병,노래방,1,2,3,344.9,침상형,1,2024-12-01,-244.9,-4.62,4.880489,-4.62,-3.880489
3111,S20240703112,14사단,영남권,36.446305,128.239051,일병,체력단련실,1,0,1,128.1,침상형,3,2024-11-25,-28.1,-0.20,1.834207,-0.20,1.165793


### 문제 10.
- **목표**: 세 테이블의 행 수를 빠르게 비교한다.
- **설명**: `len(df)`는 행 수와 동일합니다. `shape[0]`과 같습니다. 로그(서버)·거래(군수품)·개인 단위(복지) 규모 차이를 감 잡습니다.
- **실습**: 세 데이터프레임 각각 `len(df)`로 행 수를 출력하세요.
- **힌트**: 열 수는 `len(df.columns)`입니다.


In [55]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

len(dr_srv), len(dr_sup), len(df_wel)

(11000, 11000, 11000)

# 2. 행·열 선택



## 2.1 열 이름 기반 (`loc`, `[]`)



### 문제 11.
- **목표**: `loc`으로 열 이름 기반 슬라이싱을 연습한다.
- **설명**: `loc[행조건, 열리스트]`에서 `:`는 모든 행입니다. 가독성이 좋아 실무에서 자주 씁니다.
- **실습**: `df_srv.loc[:, ['server_id', 'location', 'cpu_usage']].head()`
- **힌트**: 행을 먼저 필터한 뒤 같은 열만 쓰면 분석 파이프라인이 명확해집니다.


In [56]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.loc[:, ['location', 'check_type', 'memory_usage']]

,location,check_type,memory_usage
0,17사단,월간,30.8
1,25사단,일간,18.2
2,21사단,일간,27.9
3,18사단,일간,24.0
4,17사단,주간,34.7
...,...,...,...
10995,14사단,주간,40.6
10996,14사단,주간,30.2
10997,23사단,월간,40.3
10998,12사단,월간,26.2


### 문제 12.
- **목표**: 대괄호 `[]`로 열 부분집합을 만든다.
- **설명**: 단일 열은 Series, 열 리스트는 DataFrame을 반환합니다.
- **실습**: `df_sup[['item_name', 'category', 'stock_level']].head()`
- **힌트**: 한 열만 쓸 때는 `df['col']` 이중 대괄호가 아닙니다.


In [57]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
# 아래 문법은 지양
dr_sup[['item_name', 'category']]

,item_name,category
0,Glasses,의복
1,USB Drive,사무
2,Rice Pack,식량
3,Flashlight,사무
4,Rice Pack,식량
...,...,...
10995,Medical Gloves,의료
10996,Toilet Paper,위생
10997,Rice Pack,식량
10998,First Aid Kit,의료


### 문제 13.
- **목표**: 복지 데이터에서 식별자·지역·만족도 등 핵심 열만 추린다.
- **설명**: 민감·개인 데이터 다룰 때는 필요한 열만 선택하는 습관이 중요합니다.
- **실습**: `df_wel.loc[:, ['soldier_id', 'region', 'satisfaction_score']].head()`
- **힌트**: 나중에 `merge`할 키 열을 항상 포함했는지 확인하세요.


In [58]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

# 원본 데이터에 영향을 줄 수 있으니 .copy()를 쓰기
df_wel2 = df_wel.loc[:, ['soldier_id', 'region', 'satisfaction_score']].copy()
df_wel2.head()

,soldier_id,region,satisfaction_score
0,S20240700001,전라권,6
1,S20240700002,전라권,6
2,S20240700003,전라권,5
3,S20240700004,수도권,3
4,S20240700005,수도권,3


## 2.2 위치 기반 (`iloc`)



### 문제 14.
- **목표**: `iloc` 정수 위치 인덱싱으로 첫 행을 가져온다.
- **설명**: `iloc[0]`은 첫 번째 행(Series)입니다. 라벨 인덱스와 무관하게 ‘위치’로 접근합니다.
- **실습**: `df_srv.iloc[0, :]` 첫 행 전체
- **힌트**: 마지막 행은 `iloc[-1]`입니다.


In [59]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

# dr_srv.iloc[:, ['check_date']]  # IndexError: .iloc requires numeric indexers, got ['check_date']
# iloc 사용 시, 자동화 할 때, 적극적으로 사용
# 분석할 때는 loc를 사용하자
dr_srv.iloc[:, 7]

,cpu_usage
0,21.1
1,11.6
2,41.0
3,36.1
4,20.8
...,...
10995,30.0
10996,10.5
10997,21.4
10998,13.9


### 문제 15.
- **목표**: 행·열 범위를 슬라이싱한다.
- **설명**: `iloc[a:b, c:d]`는 파이썬 슬라이스 규칙(끝 미포함)을 따릅니다.
- **실습**: `df_sup.iloc[:5, :3]` 앞 5행·앞 3열
- **힌트**: `iloc[:, -1]`은 마지막 열 전체입니다.


In [60]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_sup.iloc[:5, :3]

,supply_id,unit_code,item_code
0,SUPP00001,31A,ITEM011
1,SUPP00002,18A,ITEM015
2,SUPP00003,19C,ITEM008
3,SUPP00004,21C,ITEM019
4,SUPP00005,40A,ITEM008


### 문제 16.
- **목표**: 컬럼명을 코드로 읽어 동적 선택한다.
- **설명**: 첫 열이 항상 ID라는 가정이 맞는지 데이터 정의서와 맞춰야 합니다.
- **실습**: 첫 번째 컬럼명을 변수에 담아 `df_wel.loc[:, [첫컬럼]].head()`
- **힌트**: `df.columns[n]`으로 n번째 열 이름을 쓸 수 있습니다.


In [61]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

fc = df_wel.columns[0]
fc

'soldier_id'

In [62]:
df_wel.loc[:, [fc]].head()

,soldier_id
0,S20240700001
1,S20240700002
2,S20240700003
3,S20240700004
4,S20240700005


In [63]:
fc = df_wel.columns.tolist()[0:5]
fc

['soldier_id', 'unit_code', 'region', 'lat', 'lon']

In [64]:
df_wel.loc[:, fc].head()

,soldier_id,unit_code,region,lat,lon
0,S20240700001,17사단,전라권,36.517251,127.006332
1,S20240700002,14사단,전라권,36.227541,127.840048
2,S20240700003,23사단,전라권,37.038748,127.239173
3,S20240700004,25사단,수도권,38.416642,128.533748
4,S20240700005,21사단,수도권,37.097390,127.730811


### 문제 17.
- **목표**: 인덱스의 일부와 특정 열만 조합한다.
- **설명**: `df.index[:10]`은 앞 10개 인덱스 라벨입니다. 정수 기본 인덱스면 0~9와 대응됩니다.
- **실습**: `df_srv.loc[df_srv.index[:10], ['log_id', 'check_type']]`
- **힌트**: 필터 후 `reset_index`하면 인덱스가 바뀔 수 있습니다.


In [65]:
dr_srv.index[:10]

RangeIndex(start=0, stop=10, step=1)

In [66]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.loc[dr_srv.index[:10], ['log_id', 'check_type']]

,log_id,check_type
0,CHK000001,월간
1,CHK000002,일간
2,CHK000003,일간
3,CHK000004,일간
4,CHK000005,주간
5,CHK000006,일간
6,CHK000007,주간
7,CHK000008,월간
8,CHK000009,일간
9,CHK000010,주간


### 문제 18.
- **목표**: 중간 구간 행을 `iloc`으로 집어 본다.
- **설명**: 대용량에서 중간 행을 보면 앞·뒤와 다른 패턴(결측, 이상값)이 있는지 힌트가 됩니다.
- **실습**: `df_sup.iloc[100:105]` 100~104행 전체 열
- **힌트**: 범위를 바꿔 여러 구간을 샘플링해 보세요.


In [67]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_sup.iloc[100:105, :]

,supply_id,unit_code,item_code,item_name,category,stock_level,avg_daily_usage,weekend_usage_ratio,delivery_lead_time,requested_quantity,reorder_threshold,auto_reorder_flag,request_date,is_weekend,is_weekend_check
100,SUPP00101,36B,ITEM002,Medical Gloves,의료,40.304752,1.18461,0.18,2,35,15,0,2024-03-13,0,False
101,SUPP00102,15A,ITEM005,Notebook,사무,11.000000,2.30000,0.14,7,15,20,1,2024-04-11,0,False
102,SUPP00103,11A,ITEM013,Blanket,의복,19.000000,4.00000,0.29,6,31,20,1,2024-04-03,0,False
103,SUPP00104,18A,ITEM016,Printer Ink,사무,25.000000,5.20000,0.30,5,31,20,1,2024-03-09,1,True
104,SUPP00105,11A,ITEM016,Printer Ink,사무,16.000000,3.40000,0.47,6,17,20,1,2024-03-05,0,False


# 3. 조건 필터링



## 3.1 단일 조건



### 문제 19.
- **목표**: 단일 조건으로 행을 필터링한다.
- **설명**: 불리언 마스크 `df['col'] == 값`을 `loc`의 첫 인자에 넣습니다.
- **실습**: `issue_category`가 `'디스크'`인 서버 점검 행만 `loc`으로.
- **힌트**: 문자열은 대소문자·공백까지 정확히 일치해야 합니다.


In [68]:
dr_srv['issue_category'] == '디스크'

,issue_category
0,True
1,False
2,False
3,False
4,False
...,...
10995,True
10996,True
10997,False
10998,False


In [69]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
# dr_srv.loc[행, 열]
# .reset_index(drop = True) : index 정렬
dr_srv2 = dr_srv.loc[dr_srv['issue_category'] == '디스크', :].reset_index(drop = True).copy()
dr_srv2

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
0,CHK000001,SRV0001,17사단,2024-12-11 05:00:00,월간,0,디스크,21.1,30.8,24.0,1,137,1,1,1.20
1,CHK000009,SRV0009,18사단,2024-10-31 12:00:00,일간,1,디스크,9.4,36.6,11.1,0,321,0,0,3.56
2,CHK000010,SRV0010,13사단,2024-09-18 13:00:00,주간,0,디스크,40.2,18.8,23.4,0,155,1,0,1.48
3,CHK000015,SRV0015,11사단,2024-08-05 06:00:00,일간,1,디스크,21.9,16.3,17.1,1,143,0,0,2.94
4,CHK000017,SRV0017,22사단,2024-08-28 05:00:00,월간,1,디스크,50.6,33.2,22.9,1,94,1,0,2.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2707,CHK010988,SRV0188,25사단,2024-08-31 04:00:00,일간,0,디스크,19.1,35.2,20.2,1,288,1,1,0.66
2708,CHK010994,SRV0194,11사단,2024-10-25 14:00:00,주간,1,디스크,17.1,30.5,24.1,1,213,1,1,0.96
2709,CHK010995,SRV0195,14사단,2024-11-01 21:00:00,주간,0,디스크,35.0,16.2,22.4,0,273,0,1,2.86
2710,CHK010996,SRV0196,14사단,2024-12-30 15:00:00,주간,0,디스크,30.0,40.6,52.1,1,97,1,0,6.18


### 문제 20.
- **목표**: 군수품 범주(`category`)로 서브셋을 만든다.
- **설명**: 품목군별로 재고 정책이 다를 수 있어 범주 필터는 자주 씁니다.
- **실습**: `category`가 `'의료'`인 군수품 행만.
- **힌트**: 고유 범주는 `df['category'].unique()`로 확인할 수 있습니다.


In [70]:
dr_sup['category'].unique()

array(['의복', '사무', '식량', '의료', '위생'], dtype=object)

In [71]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_sup2 = dr_sup.loc[dr_sup['category'] == '의료', :].reset_index(drop=True).copy()
dr_sup2.head()


,supply_id,unit_code,item_code,item_name,category,stock_level,avg_daily_usage,weekend_usage_ratio,delivery_lead_time,requested_quantity,reorder_threshold,auto_reorder_flag,request_date,is_weekend,is_weekend_check
0,SUPP00007,19C,ITEM018,Bandage,의료,29.000000,5.900000,0.43,6,38,10,1,2024-03-25,0,False
1,SUPP00010,15A,ITEM009,First Aid Kit,의료,14.000000,2.900000,0.47,7,20,16,1,2024-05-07,0,False
2,SUPP00032,29B,ITEM018,Bandage,의료,27.000000,5.600000,0.37,3,36,10,1,2024-06-07,0,False
3,SUPP00033,15A,ITEM018,Bandage,의료,20.000000,4.100000,0.37,7,26,10,1,2024-05-12,1,True
4,SUPP00042,15A,ITEM002,Medical Gloves,의료,44.234419,1.114907,0.27,7,10,15,0,2024-02-12,0,False


In [72]:
import seaborn as sns
tips = sns.load_dataset("tips")

# 문제, time이 Dinner인 것만 조회
tips2 = tips.loc[tips['time'] == 'Dinner', :].reset_index(drop=True).copy()
tips2

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
171,29.03,5.92,Male,No,Sat,Dinner,3
172,27.18,2.00,Female,Yes,Sat,Dinner,2
173,22.67,2.00,Male,Yes,Sat,Dinner,2
174,17.82,1.75,Male,No,Sat,Dinner,2


### 문제 21.
- **목표**: 최빈값(또는 특정) 범주로 필터링한다.
- **설명**: `value_counts().index[0]`은 가장 많이 나온 `region`입니다. 지역 편중 여부를 볼 때 유용합니다.
- **실습**: `region`에서 가장 빈도가 높은 값 하나를 골라, 해당 지역 행만 필터링.
- **힌트**: 특정 지역명을 알고 있다면 `== '지역명'`으로 바꿔도 됩니다.


In [73]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

type(df_wel['region'])  # pandas.core.series.Series

df_wel['region'].value_counts() # SELECT region, COUNT(*) FROM df_wel GROUP BY region ORDER BY 2 DESC;

top_r = df_wel['region'].value_counts().index[0]  # 가장 빈도가 높은 value 추출

# 빅데이터 분석기사 1과목 pandas 문제 예시
df_wel.loc[df_wel['region'] == top_r, :].head()

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
13,S20240700014,21사단,충청권,37.313209,128.155869,이병,독서실,0,1,6,255.2,격리형,3,2024-12-16,-155.2,-0.88,2.453689,-0.38,0.546311
14,S20240700015,18사단,충청권,36.757347,127.095186,이병,노래방,1,1,1,241.7,침상형,2,2024-08-02,-141.7,-3.04,0.443204,-3.04,-2.113591
15,S20240700016,15사단,충청권,36.151729,128.226334,이병,상담실,0,1,5,201.0,격리형,3,2024-10-05,-101.0,-0.02,1.906278,0.48,1.093722
16,S20240700017,14사단,충청권,38.132633,128.128031,일병,상담실,0,2,1,330.0,침상형,1,2024-10-19,-230.0,-5.25,0.272409,-5.25,-4.759858
19,S20240700020,13사단,충청권,37.353614,129.294437,상병,상담실,0,5,5,407.5,침상형,2,2024-11-12,-307.5,-5.19,0.711786,-4.69,-4.031971


In [74]:
import seaborn as sns
tips = sns.load_dataset("tips")

# 1. size의 빈도수가 가장 적은 것만 추출하세요
bottom_size = tips['size'].value_counts().index[-1]

# 2. 빈도수가 가장 적은 사이즈를 기준으로 tips 데이터를 추출하세요
tips.loc[tips['size'] == bottom_size, :]

,total_bill,tip,sex,smoker,day,time,size
125,29.80,4.2,Female,No,Thur,Lunch,6
141,34.30,6.7,Male,No,Thur,Lunch,6
143,27.05,5.0,Female,No,Thur,Lunch,6
156,48.17,5.0,Male,No,Sun,Dinner,6


In [75]:
import seaborn as sns
tips = sns.load_dataset("tips")

# total_bill >= 30인 것만 조회
tips.loc[tips['total_bill'] >= 30, :].head()

,total_bill,tip,sex,smoker,day,time,size
11,35.26,5.00,Female,No,Sun,Dinner,4
23,39.42,7.58,Male,No,Sat,Dinner,4
39,31.27,5.00,Male,No,Sat,Dinner,3
44,30.40,5.60,Male,No,Sun,Dinner,4
47,32.40,6.00,Male,No,Sun,Dinner,4


## 3.2 복합 조건·`isin`·비교



### 문제 22.
- **목표**: AND(`&`)로 두 조건을 동시에 만족하는 행만 남긴다.
- **설명**: 각 조건은 괄호로 감싸야 합니다. `and` 대신 `&`를 씁니다.
- **실습**: `fix_required == 1` 이고 `issue_detected == 1` 인 행 (AND).
- **힌트**: OR는 `|` 입니다.


In [76]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

# dr_srv.loc[(조건문1) & (조건문2), :]
dr_srv.loc[(dr_srv['fix_required'] == 1) & (dr_srv['issue_detected'] == 1), :].head()

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
15,CHK000016,SRV0016,22사단,2024-11-21 05:00:00,월간,1,CPU과부하,17.5,25.3,13.3,0,204,1,1,3.14
17,CHK000018,SRV0018,정보통신대,2024-10-06 23:00:00,월간,1,CPU과부하,16.4,31.2,39.5,0,184,0,1,2.28
19,CHK000020,SRV0020,기술지원대,2024-12-02 23:00:00,월간,1,전력,29.8,24.6,34.9,0,212,0,1,2.42
22,CHK000023,SRV0023,22사단,2024-11-23 11:00:00,일간,1,없음,40.7,50.1,14.8,1,60,0,1,1.46
28,CHK000029,SRV0029,14사단,2024-12-04 05:00:00,월간,1,디스크,48.8,36.0,35.6,1,219,1,1,2.14


In [77]:
import seaborn as sns
tips = sns.load_dataset("tips")

# 레스토랑에서 웨이터에게 tip을 얼마나 주었는지 조사한 데이터
# total_bill 30 이상이면서, time이 Dinner인 것을 추출
tips.loc[(tips['total_bill'] >= 30) & (tips['time'] == 'Dinner'), :].head()

,total_bill,tip,sex,smoker,day,time,size
11,35.26,5.00,Female,No,Sun,Dinner,4
23,39.42,7.58,Male,No,Sat,Dinner,4
39,31.27,5.00,Male,No,Sat,Dinner,3
44,30.40,5.60,Male,No,Sun,Dinner,4
47,32.40,6.00,Male,No,Sun,Dinner,4


### 문제 23.
- **목표**: 범주 값이 여러 개 중 하나일 때 `isin`을 쓴다.
- **설명**: 긴 OR 체인 대신 리스트로 깔끔하게 씁니다.
- **실습**: `check_type`이 `'일간'` 또는 `'주간'` (`isin`).
- **힌트**: 부정은 `~df['col'].isin([...])` 입니다.


In [78]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.loc[dr_srv['check_type'].isin(['일간', '주간']), :].head(1)

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
1,CHK000002,SRV0002,25사단,2024-12-10 06:00:00,일간,1,CPU과부하,11.6,18.2,16.1,0,169,1,0,4.67


In [79]:
dr_srv.loc[~dr_srv['check_type'].isin(['일간', '주간']), :].head(1)

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
0,CHK000001,SRV0001,17사단,2024-12-11 05:00:00,월간,0,디스크,21.1,30.8,24.0,1,137,1,1,1.2


### 문제 24.
- **목표**: 이슈 유형 복수 선택으로 CPU·디스크 관련 로그만 본다.
- **설명**: 운영 이슈 유형별 SLA를 볼 때 자주 하는 패턴입니다.
- **실습**: `issue_category`가 `['CPU과부하', '디스크']` 중 하나 (`isin`).
- **힌트**: 카테고리 순서는 리스트에만 영향 없고 필터 결과는 동일합니다.


In [80]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.loc[dr_srv['issue_category'].isin(['CPU과부하', '디스크']), :].head()

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
0,CHK000001,SRV0001,17사단,2024-12-11 05:00:00,월간,0,디스크,21.1,30.8,24.0,1,137,1,1,1.20
1,CHK000002,SRV0002,25사단,2024-12-10 06:00:00,일간,1,CPU과부하,11.6,18.2,16.1,0,169,1,0,4.67
2,CHK000003,SRV0003,21사단,2024-08-07 02:00:00,일간,0,CPU과부하,41.0,27.9,20.1,1,269,1,1,4.12
4,CHK000005,SRV0005,17사단,2024-08-07 19:00:00,주간,0,CPU과부하,20.8,34.7,28.3,0,233,1,1,2.87
8,CHK000009,SRV0009,18사단,2024-10-31 12:00:00,일간,1,디스크,9.4,36.6,11.1,0,321,0,0,3.56


### 문제 25.
- **목표**: 수치 임계값으로 부하 높은 서버만 본다.
- **설명**: 임계값은 도메인 합의(예: 40%)가 필요합니다.
- **실습**: `cpu_usage`가 40 초과인 행.
- **힌트**: 경계 포함은 `>=` 로 조정합니다.


In [81]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.loc[dr_srv['cpu_usage'] > 40, :].head()

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
2,CHK000003,SRV0003,21사단,2024-08-07 02:00:00,일간,0,CPU과부하,41.0,27.9,20.1,1,269,1,1,4.12
9,CHK000010,SRV0010,13사단,2024-09-18 13:00:00,주간,0,디스크,40.2,18.8,23.4,0,155,1,0,1.48
16,CHK000017,SRV0017,22사단,2024-08-28 05:00:00,월간,1,디스크,50.6,33.2,22.9,1,94,1,0,2.28
22,CHK000023,SRV0023,22사단,2024-11-23 11:00:00,일간,1,없음,40.7,50.1,14.8,1,60,0,1,1.46
23,CHK000024,SRV0024,13사단,2024-09-25 18:00:00,일간,0,전력,43.1,28.8,30.1,1,114,1,1,2.62


### 문제 26.
- **목표**: 두 열의 행별 비교로 ‘재주문 필요’ 상황에 가까운 행을 찾는다.
- **설명**: 실제 비즈니스 룰과 다를 수 있으나 EDA 패턴으로 유용합니다.
- **실습**: `stock_level`이 `reorder_threshold` 미만인 행.
- **힌트**: 같은 행의 두 열을 비교할 때는 각각 Series이므로 브로드캐스트됩니다.


In [82]:
import seaborn as sns
iris = sns.load_dataset('iris')
# iris['sepal_width'] >= iris['petal_length']

# iris2 = iris.loc[:, ['sepal_width', 'petal_length']]
# iris2['비교'] = iris2['sepal_width'] >= iris2['petal_length']
# iris2.loc[iris2['비교'] == True]

iris.loc[iris['sepal_width'] >= iris['petal_length'], :].head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [83]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_sup.loc[dr_sup['stock_level'] < dr_sup['reorder_threshold'], # 행추출
 ['item_code', 'item_name', 'stock_level', 'reorder_threshold'] # 열추출
].head()

,item_code,item_name,stock_level,reorder_threshold
2,ITEM008,Rice Pack,19.0,20
4,ITEM008,Rice Pack,18.0,20
5,ITEM019,Flashlight,8.0,20
8,ITEM007,Shirt,16.0,20
9,ITEM009,First Aid Kit,14.0,16


### 문제 27.
- **목표**: 정신 건강 점수 구간으로 서브셋을 잡는다.
- **설명**: 구간 경계는 보고 목적에 맞게 조정합니다.
- **실습**: `mental_wellbeing_score`가 6 이상 8 이하.
- **힌트**: `between(6, 8)` 메서드로도 같은 조건을 쓸 수 있습니다.


In [84]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

df_wel.loc[(df_wel['mental_wellbeing_score'] >= 6) &
           (df_wel['mental_wellbeing_score'] <= 8), :].head()

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
5,S20240700006,18사단,강원권,36.463025,127.308953,일병,PX,1,1,4,397.4,침상형,6,2024-07-26,-297.4,-5.44,11.530000,-4.94,-5.530000
64,S20240700065,22사단,전라권,35.938120,127.625429,상병,독서실,0,2,6,418.1,격리형,6,2024-07-26,-318.1,-4.95,0.433175,-4.45,-3.626530
148,S20240700149,15사단,수도권,38.342945,128.259179,병장,체력단련실,0,4,3,204.9,격리형,8,2024-11-14,-104.9,-1.12,0.328288,-1.12,-0.107093
157,S20240700158,22사단,강원권,37.017712,127.908835,병장,노래방,0,4,4,421.5,침상형,8,2024-08-13,-321.5,-6.04,13.426039,-5.54,-5.426039
162,S20240700163,24사단,전라권,36.062218,129.033118,상병,체력단련실,0,3,1,392.9,침상형,8,2024-11-04,-292.9,-6.82,13.987459,-6.82,-5.987459


In [85]:
# pandas.Series.between
# Dataframe에서 컬럼에 접근하면 Series
# type(df_wel['mental_wellbeing_score'])  # pandas.core.series.Series
s = df_wel['mental_wellbeing_score']
s.between(1, 4).head()

,mental_wellbeing_score
0,True
1,True
2,True
3,True
4,True


In [86]:
df_wel.loc[df_wel['mental_wellbeing_score'].between(6, 8), :].head()

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
5,S20240700006,18사단,강원권,36.463025,127.308953,일병,PX,1,1,4,397.4,침상형,6,2024-07-26,-297.4,-5.44,11.530000,-4.94,-5.530000
64,S20240700065,22사단,전라권,35.938120,127.625429,상병,독서실,0,2,6,418.1,격리형,6,2024-07-26,-318.1,-4.95,0.433175,-4.45,-3.626530
148,S20240700149,15사단,수도권,38.342945,128.259179,병장,체력단련실,0,4,3,204.9,격리형,8,2024-11-14,-104.9,-1.12,0.328288,-1.12,-0.107093
157,S20240700158,22사단,강원권,37.017712,127.908835,병장,노래방,0,4,4,421.5,침상형,8,2024-08-13,-321.5,-6.04,13.426039,-5.54,-5.426039
162,S20240700163,24사단,전라권,36.062218,129.033118,상병,체력단련실,0,3,1,392.9,침상형,8,2024-11-04,-292.9,-6.82,13.987459,-6.82,-5.987459


### 문제 28.
- **목표**: 필터 후 인덱스를 0부터 다시 매긴다.
- **설명**: 후속 `concat`·저장 시 깔끔한 인덱스가 필요할 때 씁니다.
- **실습**: 필터 후 `reset_index(drop=True)`로 인덱스 초기화.
- **힌트**: `drop=False`면 이전 인덱스가 열로 남습니다.


In [87]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
# 변수명 : temp_abnormal_flag == 1
dr_srv.loc[dr_srv['temp_abnormal_flag'] == 1, :].reset_index(drop=True).head()


,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
0,CHK000001,SRV0001,17사단,2024-12-11 05:00:00,월간,0,디스크,21.1,30.8,24.0,1,137,1,1,1.20
1,CHK000003,SRV0003,21사단,2024-08-07 02:00:00,일간,0,CPU과부하,41.0,27.9,20.1,1,269,1,1,4.12
2,CHK000006,SRV0006,21사단,2024-08-12 06:00:00,일간,0,전력,14.8,33.7,8.3,1,217,0,1,1.88
3,CHK000007,SRV0007,21사단,2024-10-22 03:00:00,주간,0,전력,28.4,23.8,29.5,1,114,1,0,1.53
4,CHK000011,SRV0011,12사단,2024-07-11 01:00:00,주간,0,CPU과부하,19.1,31.6,33.8,1,141,1,1,3.61


# 4. 정렬



## 4.1 `sort_values`



### 문제 29.
- **목표**: `sort_values`로 CPU 사용률 상위 서버를 본다.
- **설명**: 상위 n개는 `head`와 조합합니다.
- **실습**: `df_srv`를 `cpu_usage` 내림차순 정렬 후 상위 8행.
- **힌트**: `na_position`으로 결측 위치를 제어할 수 있습니다.


In [88]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.sort_values('cpu_usage',ascending=False).head(8)

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
7704,CHK007705,SRV0205,24사단,2024-10-06 18:00:00,주간,0,없음,100.000000,34.000000,35.8,0,148,1,1,1.37
8760,CHK008761,SRV0061,15사단,2024-12-13 18:00:00,일간,0,전력,99.992180,84.078537,22.8,1,85,0,1,2.51
5960,CHK005961,SRV0261,기술지원대,2024-10-17 21:00:00,주간,1,없음,99.872431,98.457595,24.5,1,125,1,0,1.89
3485,CHK003486,SRV0186,13사단,2024-07-25 12:00:00,일간,0,없음,99.869435,99.915618,26.1,1,133,0,1,6.49
5337,CHK005338,SRV0238,24사단,2024-07-23 00:00:00,일간,0,전력,99.855680,87.238135,45.6,1,98,0,0,2.44
5617,CHK005618,SRV0218,15사단,2024-10-12 19:00:00,일간,0,전력,99.836476,94.962396,32.6,1,54,0,0,2.80
5938,CHK005939,SRV0239,15사단,2024-08-06 12:00:00,월간,1,전력,99.726574,82.699317,28.1,1,80,1,1,3.82
6159,CHK006160,SRV0160,22사단,2024-10-03 09:00:00,주간,1,없음,99.716177,91.362331,34.0,1,113,0,0,2.49


In [89]:
dr_srv['cpu_usage'].sort_values(ascending=False).head(8)

,cpu_usage
7704,100.000000
8760,99.992180
5960,99.872431
3485,99.869435
5337,99.855680
5617,99.836476
5938,99.726574
6159,99.716177


### 문제 30.
- **목표**: 청구 수량이 적은 순으로 정렬한다.
- **설명**: 오름차순이 기본(`ascending=True`)입니다.
- **실습**: `df_sup`를 `requested_quantity` 오름차순.
- **힌트**: 복수 열 정렬은 리스트로 열 이름을 넘깁니다.


In [90]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_sup.sort_values('requested_quantity', ascending=True).head()

,supply_id,unit_code,item_code,item_name,category,stock_level,avg_daily_usage,weekend_usage_ratio,delivery_lead_time,requested_quantity,reorder_threshold,auto_reorder_flag,request_date,is_weekend,is_weekend_check
5352,SUPP05353,40A,ITEM005,Notebook,사무,3.000000,0.800000,0.38,3,10,20,1,2024-02-28,0,False
9387,SUPP09388,12C,ITEM002,Medical Gloves,의료,40.221063,1.256695,0.19,4,10,15,0,2024-03-12,0,False
770,SUPP00771,17C,ITEM002,Medical Gloves,의료,42.457915,1.198179,0.46,4,10,15,0,2024-05-12,1,True
778,SUPP00779,15A,ITEM019,Flashlight,사무,12.000000,2.600000,0.35,7,10,20,1,2024-03-20,0,False
784,SUPP00785,17C,ITEM006,Pencil,사무,6.000000,1.400000,0.21,4,10,20,1,2024-05-10,0,False


### 문제 31.
- **목표**: 시설까지 거리가 먼 순으로 복지 레코드를 본다.
- **설명**: 접근성 불균형 분석의 출발점입니다.
- **실습**: `df_wel`를 `distance_to_facility` 내림차순.
- **힌트**: 지리 좌표(`lat`,`lon`)와 함께 보면 지도 시각화로 이어질 수 있습니다.


In [91]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

df_wel.sort_values('distance_to_facility', ascending=False).head()

,soldier_id,unit_code,region,lat,lon,rank,facility_type,facility_usage,usage_frequency_per_week,satisfaction_score,distance_to_facility,has_private_space,mental_wellbeing_score,date,facility_access_score,expected_mental_score,mental_gap_score,expected_mental_score_adj,expected_mental_score_v4
8079,S20240708080,17사단,강원권,35.839561,126.768356,상병,상담실,1,3,1,499.9,침상형,1,2024-08-01,-399.9,-9.50,1.229924,-9.50,-10.060000
9675,S20240709676,20사단,영남권,38.352427,129.175713,이병,노래방,1,1,5,499.9,격리형,10,2024-07-11,-399.9,-7.50,1.246258,-7.00,-7.660000
7532,S20240707533,12사단,강원권,37.068729,128.600727,이병,PX,0,1,1,499.9,격리형,1,2024-08-01,-399.9,-9.50,1.111512,-9.50,-10.060000
5854,S20240705855,19사단,충청권,37.186167,127.771586,병장,노래방,0,4,1,499.7,격리형,1,2024-08-23,-399.7,-9.49,0.012514,-9.49,-10.060000
1117,S20240701118,23사단,충청권,36.691860,126.897135,병장,상담실,0,4,1,499.7,침상형,2,2024-09-01,-399.7,-9.49,0.314939,-9.49,-9.281759


### 문제 32.
- **목표**: 다중 키 정렬로 같은 부대 내 시간 순서 등을 맞춘다.
- **설명**: 먼저 쓴 열이 1차 정렬 키입니다.
- **실습**: `location`, `check_date` 기준 다중 정렬 (오름차순).
- **힌트**: `ascending=[True, False]`처럼 열마다 다르게 줄 수 있습니다.


In [92]:
# ── Pandas 실습 ─────────────────────────────────────────────────────

dr_srv.sort_values(['location', 'check_date'], ascending=[True, False]).head()

,log_id,server_id,location,check_date,check_type,issue_detected,issue_category,cpu_usage,memory_usage,disk_usage,temp_abnormal_flag,uptime_days,auto_alert_triggered,fix_required,fix_duration_hours
10798,CHK010799,SRV0299,11사단,2024-12-30 12:00:00,주간,1,CPU과부하,20.900000,46.800000,50.7,1,203,1,0,6.31
1524,CHK001525,SRV0025,11사단,2024-12-30 03:00:00,일간,0,CPU과부하,37.800000,37.300000,58.4,1,108,0,0,2.80
5665,CHK005666,SRV0266,11사단,2024-12-29 20:00:00,주간,1,없음,90.817147,92.764464,14.9,1,274,1,0,2.90
7886,CHK007887,SRV0087,11사단,2024-12-29 16:00:00,월간,1,없음,39.400000,28.200000,58.7,0,92,0,0,3.72
156,CHK000157,SRV0157,11사단,2024-12-29 08:00:00,주간,0,없음,41.100000,38.400000,30.1,1,204,1,1,3.61


# 5. 그룹 집계



## 5.1 `groupby`·`agg`



### 문제 33.
- **목표**: `groupby`로 부대(`location`)별 점검 건수를 센다.
- **설명**: 집계 함수로 `count`,`size` 등을 씁니다. `count`는 비결측만 셉니다.
- **실습**: `location`별 행 수 (`log_id` count).
- **힌트**: `size()`는 그룹 크기(행 수)로 결측과 무관합니다.


In [93]:
tips.groupby('size')['total_bill'].sum()

,total_bill
size,
1,28.97
2,2565.89
3,884.55
4,1058.70
5,150.34
6,139.32


In [94]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

dr_srv.groupby('location')['log_id'].count()

,log_id
location,
11사단,638
12사단,636
13사단,645
14사단,643
15사단,639
16사단,631
17사단,627
18사단,641
19사단,616


In [95]:
# tips 데이터
# 요일별 size의 개수 구하기

tips.groupby('day')['size'].count()

/tmp/ipykernel_2921/3009490562.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby('day')['size'].count()


,size
day,
Thur,62
Fri,19
Sat,87
Sun,76


In [96]:
tips.groupby(['day', 'sex'])['size'].count()  # 시리즈
# 시리즈의 구성 요소 : index와 value를 가진 것
# day와 sex가 index

/tmp/ipykernel_2921/3723579971.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby(['day', 'sex'])['size'].count()  # 시리즈


day   sex   
Thur  Male      30
      Female    32
Fri   Male      10
      Female     9
Sat   Male      59
      Female    28
Sun   Male      58
      Female    18
Name: size, dtype: int64

### 문제 34.
- **목표**: 이슈 유형별 평균 CPU를 비교한다.
- **설명**: 그룹별 대표값 비교는 운영 인사이트의 기본입니다.
- **실습**: `issue_category`별 `cpu_usage` 평균.
- **힌트**: 같은 패턴으로 `median`,`std`도 가능합니다.


In [97]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

dr_srv.groupby('issue_category')['cpu_usage'].mean()

,cpu_usage
issue_category,
CPU과부하,28.755990
디스크,28.518417
없음,29.198480
전력,28.985871


In [98]:
dr_srv.groupby('issue_category')['cpu_usage'].min().sort_values(ascending=False)

,cpu_usage
issue_category,
디스크,3.9
전력,3.9
CPU과부하,1.7
없음,0.0


### 문제 35.
- **목표**: `agg`로 그룹당 여러 통계를 한 번에 구한다.
- **설명**: 점검 주기별로 수리 시간 평균·최대를 동시에 봅니다.
- **실습**: `check_type`별 `fix_duration_hours` 평균·최대 (`agg`).
- **힌트**: 딕셔너리로 열마다 다른 함수를 줄 수도 있습니다.


In [99]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

dr_srv.groupby('check_type')['fix_duration_hours'].agg(['mean', 'max', 'min', 'sum'])

,mean,max,min,sum
check_type,,,,
월간,2.983514,11.56,0.14,10970.38
일간,3.016626,11.79,0.21,11004.65
주간,2.998414,11.47,0.25,11019.17


### 문제 36.
- **목표**: 품목 범주별 청구량 합계와 평균을 동시에 본다.
- **설명**: 소계·평균을 같이 보면 ‘총량은 크지만 건당 작다’ 같은 패턴이 보입니다.
- **실습**: `category`별 `requested_quantity` 합·평균.
- **힌트**: `groupby` 후 `transform`이면 원본 행에 붙이기도 합니다.


In [100]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

dr_sup.groupby('category')['requested_quantity'].agg(['sum', 'mean'])

,sum,mean
category,,
사무,87473,30.898269
식량,67139,41.546411
위생,113059,41.673056
의료,41211,24.946126
의복,67282,30.750457


### 문제 37.
- **목표**: 부대 코드별 평균 재고 수준 상위를 본다.
- **설명**: 자동발주·물류 우선순위 논의에 쓸 수 있습니다.
- **실습**: `unit_code`별 `stock_level` 평균 상위 10개 부대.
- **힌트**: 건수가 적은 부대는 평균이 왜곡될 수 있어 `count`와 함께 보세요.


In [101]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

dr_sup.groupby('unit_code')['stock_level'].agg(['mean', 'count']).sort_values('unit_code', ascending=False).head()

,mean,count
unit_code,,
40A,22.980053,687
39C,23.260066,758
36B,22.891456,724
33B,22.710323,718
31A,22.753117,739


In [102]:
dr_sup.groupby('unit_code')['stock_level'].mean().sort_values(ascending=False).head()

,stock_level
unit_code,
12C,24.259959
17C,23.835893
21C,23.702343
18C,23.465417
11A,23.353188


### 문제 38.
- **목표**: 시설 유형별 만족도 평균을 비교한다.
- **설명**: 정책별(체육관·독서실 등) 만족도 차이를 가설 검정 전에 봅니다.
- **실습**: `facility_type`별 `satisfaction_score` 평균.
- **힌트**: 표본 수가 다르면 신뢰구간·검정이 필요합니다.


In [103]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

df_wel.groupby('facility_type')['satisfaction_score'].mean().head()

,satisfaction_score
facility_type,
PX,3.399052
노래방,3.413448
독서실,3.379088
상담실,3.476656
체력단련실,3.418242


### 문제 39.
- **목표**: 계급(`rank`)별 정신건강 점수 중앙값을 본다.
- **설명**: 이상치에 덜 민감한 대표값입니다.
- **실습**: `rank`별 `mental_wellbeing_score` 중앙값.
- **힌트**: `mean`과 `median`을 나란히 비교하면 분포 꼬리를 짐작합니다.


In [104]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환

df_wel.groupby('rank')['mental_wellbeing_score'].median().head()

,mental_wellbeing_score
rank,
병장,2.0
상병,2.0
이병,2.0
일병,1.0


### 문제 40.
- **목표**: 지역×시설 유형 교차로 만족도를 본다.
- **설명**: 다차원 그룹은 `groupby([col1,col2])` 입니다.
- **실습**: `region`·`facility_type` 다중 그룹 평균 만족도.
- **힌트**: 결과가 길면 `unstack`으로 피벗 형태를 만들 수 있습니다.


In [105]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환
# 컬럼 : satisfaction_score

df_wel.groupby(['region', 'facility_type'])['satisfaction_score'].mean()

region  facility_type
강원권     PX               3.378378
        노래방              3.420091
        독서실              3.365957
        상담실              3.504566
        체력단련실            3.476886
수도권     PX               3.349882
        노래방              3.458050
        독서실              3.350622
        상담실              3.476404
        체력단련실            3.431925
영남권     PX               3.373434
        노래방              3.435484
        독서실              3.422269
        상담실              3.578366
        체력단련실            3.393805
전라권     PX               3.369412
        노래방              3.421053
        독서실              3.455947
        상담실              3.417943
        체력단련실            3.452785
충청권     PX               3.513158
        노래방              3.334812
        독서실              3.298643
        상담실              3.407895
        체력단련실            3.338164
Name: satisfaction_score, dtype: float64

In [106]:
result = df_wel.groupby(['region', 'facility_type'])['satisfaction_score'].mean().unstack()
result

facility_type,PX,노래방,독서실,상담실,체력단련실
region,,,,,
강원권,3.378378,3.420091,3.365957,3.504566,3.476886
수도권,3.349882,3.458050,3.350622,3.476404,3.431925
영남권,3.373434,3.435484,3.422269,3.578366,3.393805
전라권,3.369412,3.421053,3.455947,3.417943,3.452785
충청권,3.513158,3.334812,3.298643,3.407895,3.338164


In [107]:
result.loc["강원권", "독서실"]

np.float64(3.3659574468085105)

### 문제 41.
- **목표**: `groupby().size()`로 플래그별 건수를 센다.
- **설명**: `value_counts`와 유사하지만 그룹 객체 체인에 맞출 때 씁니다.
- **실습**: `auto_reorder_flag`별 건수 (`groupby` size).
- **힌트**: `as_index=False`로 표 형태 DataFrame을 만들 수 있습니다.


In [108]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 42.
- **목표**: 이슈 탐지 여부별로 수리 필요 플래그 평균을 본다.
- **설명**: 라벨 간 연관(간단한 비율)을 탐색합니다.
- **실습**: `issue_detected`별 `fix_required` 평균.
- **힌트**: 0~1 평균은 비율 해석이 가능합니다.


In [109]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



# 6. 데이터 품질과 파생



## 6.1 결측·중복



### 문제 43.
- **목표**: 열 단위 결측 개수로 데이터 품질을 점검한다.
- **설명**: `isna().sum()`은 열마다 결측 수입니다.
- **실습**: `df_srv.isna().sum()` 결측 개수 열별.
- **힌트**: 전체 결측은 `df.isna().sum().sum()` 입니다.


- 조건필터링, 그룹바이 연산, 결측데이터 처리

In [143]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
dr_srv.isna().sum()

,0
log_id,0
server_id,0
location,0
check_date,0
check_type,0
issue_detected,0
issue_category,0
cpu_usage,0
memory_usage,0
disk_usage,0


In [144]:
import seaborn as sns
titanic = sns.load_dataset('titanic')

titanic.isna().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [145]:
# 결측치 있는 컬럼만 추출
null_cols = titanic.columns[titanic.isnull().any()]
null_cols

Index(['age', 'embarked', 'deck', 'embark_town'], dtype='object')

In [146]:
titanic[null_cols].isna().sum()

,0
age,177
embarked,2
deck,688
embark_town,2


In [148]:
# age 컬럼에서 결측치가 존재하는 행 별도로 추출
titanic.loc[titanic['age'].isnull(), :]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
17,1,2,male,NaN,0,0,13.0000,S,Second,man,True,NaN,Southampton,yes,True
19,1,3,female,NaN,0,0,7.2250,C,Third,woman,False,NaN,Cherbourg,yes,True
26,0,3,male,NaN,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,no,True
28,1,3,female,NaN,0,0,7.8792,Q,Third,woman,False,NaN,Queenstown,yes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,0,3,male,NaN,0,0,7.2292,C,Third,man,True,NaN,Cherbourg,no,True
863,0,3,female,NaN,8,2,69.5500,S,Third,woman,False,NaN,Southampton,no,False
868,0,3,male,NaN,0,0,9.5000,S,Third,man,True,NaN,Southampton,no,True
878,0,3,male,NaN,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True


In [149]:
# age 컬럼에서 결측치가 존재하지 않는 행 별도로 추출
titanic.loc[titanic['age'].notnull(), :]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,NaN,Queenstown,no,False
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [150]:
titanic.loc[~titanic['age'].isnull(), :].head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### 문제 44.
- **목표**: 수치 열 결측을 중앙값으로 채워 단순 대체를 연습한다.
- **설명**: 실무에서는 KNN·모델 기반 대체 등 더 정교한 방법도 고려합니다.
- **실습**: `df_wel` 숫자 열 결측을 열 중앙값으로 채운 뒤 남은 결측 총합.
- **힌트**: 범주형 결측은 `fillna('Unknown')` 등 별도 처리가 필요합니다.


In [160]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
df = titanic.copy()
# df['deck'] = df['deck'].fillna('ABC') # 'deck'는 category라서 에러 발생
# TypeError: Cannot setitem on a Categorical with a new category (ABC), set the categories first

# 방법 1 : 카테고리 등록 (복잡) (잘 활용하지는 않음)
# 방법 2 : 데이터 타입을 object로 변경

# 데이터 타입을 category => object로 변환
print(df.info())
df['deck'] = df['deck'].astype('object')

print("--변경--")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
None
--변경--
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to

In [162]:
df = titanic.copy()
df['deck'] = df['deck'].astype('object')
df['deck'] = df['deck'].fillna('ABC')
df.head(1)

# 문자를 처리하는 방식 => 최빈값 사용 .mode() 사용
# 숫자를 처리하는 방식 => 평균/중앙값 사용
# 결측치가 존재하는 행 삭제 => 기본적으로 비추

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.25,S,Third,man,True,ABC,Southampton,no,False


In [175]:
df = titanic.copy()
df['deck'] = df['deck'].astype('object')
df['deck'].mode()

,deck
0,C


In [176]:
df['deck'].mode()[0]
# .mode()[0]에서 [0]을 붙이는 이유
## .mode()만 하면 시리즈를 반환하기 때문에 시리즈의 첫 번째 값만 가져오기 위해

'C'

In [177]:
df['deck'] = df['deck'].fillna(df['deck'].mode()[0])
df.head(1)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.25,S,Third,man,True,C,Southampton,no,False


In [178]:
df['age'].mean(), df['age'].median()

(np.float64(29.69911764705882), 28.0)

In [179]:
df['age'] = df['age'].fillna(df['age'].median())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          891 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         891 non-null    object  
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(1), float64(2), int64(4), object(6)
memory usage: 86.4+ KB


### 문제 45.
- **목표**: 완전 중복 행 개수를 센다.
- **설명**: 모든 열이 동일한 행이 몇 개인지 봅니다.
- **실습**: `df_sup.duplicated().sum()` 전체 중복 행 개수.
- **힌트**: 특정 열 기준은 `subset=[...]` 입니다.


In [112]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 46.
- **목표**: 기본 키 후보(`supply_id`)로 중복을 제거한 유일 행 수를 본다.
- **설명**: 실제 PK 정의와 일치하는지 검증하는 단계입니다.
- **실습**: `supply_id` 기준 중복 제거 후 행 수.
- **힌트**: `keep='last'` 등으로 어떤 행을 남길지 정할 수 있습니다.


In [113]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



## 6.2 파생 변수·구간화
- 파생 변수를 잘 만드는 사람이 실력자
- 파생 변수를 만든다 => 수식을 만든다


### 문제 47.
- **목표**: 여러 부하 지표를 더한 파생 열을 만든다.
- **설명**: `assign`은 새 DataFrame을 반환하므로 원본 보존에 유리합니다.
- **실습**: 파생: `usage_sum = cpu_usage + memory_usage + disk_usage` (`assign`).
- **힌트**: 가중합이 필요하면 `w1*cpu + w2*mem` 형태로 확장합니다.


In [182]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
# tips['새로운 컬럼'] = tips['total_bill'] / tips['size']
tips['인당결제금액2'] = tips['total_bill'] / tips['size']
tips

,total_bill,tip,sex,smoker,day,time,size,인당결제금액,인당결제금액2
0,16.99,1.01,Female,No,Sun,Dinner,2,8.495000,8.495000
1,10.34,1.66,Male,No,Sun,Dinner,3,3.446667,3.446667
2,21.01,3.50,Male,No,Sun,Dinner,3,7.003333,7.003333
3,23.68,3.31,Male,No,Sun,Dinner,2,11.840000,11.840000
4,24.59,3.61,Female,No,Sun,Dinner,4,6.147500,6.147500
...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,9.676667,9.676667
240,27.18,2.00,Female,Yes,Sat,Dinner,2,13.590000,13.590000
241,22.67,2.00,Male,Yes,Sat,Dinner,2,11.335000,11.335000
242,17.82,1.75,Male,No,Sat,Dinner,2,8.910000,8.910000


In [183]:
tips = tips.drop(columns=['인당결제금액2'])
tips.head()

,total_bill,tip,sex,smoker,day,time,size,인당결제금액
0,16.99,1.01,Female,No,Sun,Dinner,2,8.495000
1,10.34,1.66,Male,No,Sun,Dinner,3,3.446667
2,21.01,3.50,Male,No,Sun,Dinner,3,7.003333
3,23.68,3.31,Male,No,Sun,Dinner,2,11.840000
4,24.59,3.61,Female,No,Sun,Dinner,4,6.147500


In [193]:
dr_srv.assign(usage_sum = dr_srv['cpu_usage'] + dr_srv['memory_usage'] + dr_srv['disk_usage']
              )[['cpu_usage', 'disk_usage', 'memory_usage', 'usage_sum']].head()

,cpu_usage,disk_usage,memory_usage,usage_sum
0,21.1,24.0,30.8,75.9
1,11.6,16.1,18.2,45.9
2,41.0,20.1,27.9,89.0
3,36.1,23.1,24.0,83.2
4,20.8,28.3,34.7,83.8


### 문제 48.
- **목표**: 연속 비율을 구간 범주로 나눈다.
- **설명**: `pd.cut`은 구간 경계와 라벨을 정의합니다.
- **실습**: `weekend_usage_ratio`를 `pd.cut`으로 low/mid/high 구간 빈도.
- **힌트**: 구간을 `qcut`으로 분위수 기준으로 나눌 수도 있습니다.


In [192]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
pd.cut(dr_sup['weekend_usage_ratio'], bins=[0, 0.25, 0.5, 1.0], labels=
       ['low', 'mid', 'high']).value_counts()


,count
weekend_usage_ratio,
mid,6650
low,4350
high,0


# 7. 문자열 처리



## 7.1 `str` 접근자



### 문제 49.
- **목표**: 품목명 문자열을 대문자로 통일한다.
- **설명**: JOIN·중복 제거 전 정규화에 쓰입니다.
- **실습**: `df_sup['item_name'].str.upper().head()`
- **힌트**: `str.lower`, `str.title`도 같은 계열입니다.


In [196]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
# dr_sup['item_name'].upper()
## AttributeError: 'Series' object has no attribute 'upper'

dr_sup['item_name'].str.upper().head()

,item_name
0,GLASSES
1,USB DRIVE
2,RICE PACK
3,FLASHLIGHT
4,RICE PACK


### 문제 50.
- **목표**: `str.contains`로 문자열 패턴 포함 여부를 센다.
- **설명**: `na=False`로 결측은 False 처리해 연산 오류를 막습니다.
- **실습**: `df_srv['location'].str.contains('사단', na=False)` True 개수.
- **힌트**: 정규식은 `regex=True`(기본)와 패턴 문자열을 사용합니다.


In [208]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
dr_srv['location'].value_counts()

,count
location,
23사단,714
정보통신대,704
기술지원대,679
25사단,678
22사단,666
13사단,645
14사단,643
20사단,642
18사단,641


In [209]:
dr_srv['location'].str.contains('사단', na=False).sum()

np.int64(9617)

### 문제 51.
- **목표**: 부대 코드에서 앞 두 글자만 잘라 상위 체계를 만든다.
- **설명**: 문자열 Series에 `str` 접근자를 씁니다.
- **실습**: `df_wel['unit_code']`를 문자열로 바꾼 뒤 앞 두 글자.
- **힌트**: 숫자형이면 `astype(str)`이 안전합니다.


In [204]:
num = str(100000)
num[:2]

'10'

In [202]:
# ── dtype 변환 ──────────────────────────────────────────────────────
# `astype`으로 자료형 변경
# 숫자 컬럼, 100000 데이터가 있다고 가정
df_wel['unit_code'].str[:2].head()


,unit_code
0,17
1,14
2,23
3,25
4,21


In [205]:
df_wel['unit_code'].astype(str).str[:2].head()

,unit_code
0,17
1,14
2,23
3,25
4,21


### 문제 52.
- **목표**: 범주 이름 길이의 평균으로 명명 패턴을 대략 본다.
- **설명**: `str.len()`은 문자 수입니다.
- **실습**: `category` 문자열 길이 `str.len()` 평균.
- **힌트**: 바이트 길이가 아닌 유니코드 문자 기준입니다.


In [207]:
# ── Pandas 실습 ─────────────────────────────────────────────────────
dr_sup['item_name'].str.len()


,item_name
0,7
1,9
2,9
3,10
4,9
...,...
10995,14
10996,12
10997,9
10998,13


# 8. 시계열 다루기



## 8.1 변환·추출·필터·집계



### 문제 53.
- **목표**: 문자열로 된 점검 시각을 datetime으로 바꾼다.
- **설명**: 시계열 연산·정렬·리샘플의 전제입니다.
- **실습**: `check_date`를 `pd.to_datetime` 변환 후 dtype 확인.
- **힌트**: `errors='coerce'`로 깨진 값은 NaT가 됩니다.


In [120]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 54.
- **목표**: 점검 연도별 건수로 연도 편중을 본다.
- **설명**: `.dt` 접근자는 datetime Series에만 사용합니다.
- **실습**: 점검일에서 연도 `.dt.year` 빈도.
- **힌트**: 월별은 `dt.month`, 분기는 `dt.quarter`입니다.


In [121]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 55.
- **목표**: 청구일을 월 단위로 집계한다.
- **설명**: `to_period('M')`은 월 구간 라벨을 만듭니다.
- **실습**: `request_date` datetime 변환 후 월별 건수.
- **힌트**: `resample`은 인덱스가 DatetimeIndex일 때 특히 편합니다.


In [122]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 56.
- **목표**: 특정 일 이후의 서버 점검만 남긴다.
- **설명**: 문자열과 비교해도 pandas가 datetime으로 맞춰 줄 때가 많습니다.
- **실습**: 2024-10-01 이후 서버 점검만 필터.
- **힌트**: 구간은 `between`으로도 표현할 수 있습니다.


In [123]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 57.
- **목표**: 기록 일자의 요일 분포를 본다.
- **설명**: `dayofweek`는 월=0 … 일=6 (pandas 기본).
- **실습**: `df_wel['date']`의 요일(`dayofweek`) 빈도.
- **힌트**: 요일 이름 매핑은 `map`으로 붙일 수 있습니다.


In [124]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 58.
- **목표**: 월별 이슈 탐지 건수 합으로 추세를 본다.
- **설명**: 파생 월 열을 만든 뒤 `groupby`합니다.
- **실습**: 월별 `issue_detected` 합.
- **힌트**: 결측 날짜 행은 제외되거나 NaT 그룹이 생길 수 있습니다.


In [125]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 59.
- **목표**: 청구일 순으로 데이터를 정렬한다.
- **설명**: datetime 열을 만든 뒤 `sort_values`합니다.
- **실습**: `request_date` 기준 정렬.
- **힌트**: 시간대가 섞이면 `tz_localize` 등이 필요할 수 있습니다.


In [126]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 60.
- **목표**: 점검 기간의 최소·최대 날짜로 데이터 범위를 확인한다.
- **설명**: 리포트의 ‘분석 기간’ 문구에 씁니다.
- **실습**: `check_date` 최소·최대.
- **힌트**: `idxmin`으로 최소 날짜가 있는 행 인덱스도 찾을 수 있습니다.


In [127]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



# 9. 테이블 결합·변형



## 9.1 `merge`·`concat`·`pivot_table`



### 문제 61.
- **목표**: 군수품과 복지 데이터를 `unit_code`로 내부 조인한다.
- **설명**: inner는 양쪽 모두 키가 있는 행만 남깁니다. 행 수 폭증에 주의합니다.
- **실습**: `df_sup`와 `df_wel`을 `unit_code`로 inner merge 후 `shape`.
- **힌트**: `suffixes`로 겹치는 열 이름을 구분합니다.


In [128]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 62.
- **목표**: 조인 결과에서 품목 범주×지역 교차 빈도를 본다.
- **설명**: 다대다 조인이면 행이 늘어난 ‘중복 의미’를 이해해야 합니다.
- **실습**: `unit_code`로 merge한 뒤 `category`·`region`별 행 수 (단일 셀에서 완료).
- **힌트**: 필요하면 `drop_duplicates`로 단위를 맞춥니다.


In [129]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 63.
- **목표**: 집계 테이블을 원본에 붙여 위치별 평균 CPU를 각 행에 표시한다.
- **설명**: SQL의 윈도 없이 ‘부대 평균’을 열로 붙이는 패턴입니다.
- **실습**: `location`별 평균 `cpu_usage`를 구해 원본 `df_srv`에 `merge`로 붙이기.
- **힌트**: 같은 `location`이면 같은 평균이 반복됩니다.


In [130]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 64.
- **목표**: `concat`으로 행 방향으로 이어 붙인다.
- **설명**: axis=0이 세로 결합입니다. 인덱스가 중복될 수 있습니다.
- **실습**: `pd.concat`으로 `df_srv`의 0~1행과 2~3행을 세로 결합.
- **힌트**: `ignore_index=True`로 인덱스를 새로 매길 수 있습니다.


In [131]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 65.
- **목표**: `pivot_table`로 범주별 대표값을 표 형태로 만든다.
- **설명**: 엑셀 피벗과 대응됩니다.
- **실습**: `category`별 평균 `avg_daily_usage` (`pivot_table`).
- **힌트**: `columns` 인자로 넓은 표를 만들 수 있습니다.


In [132]:
# ── 피벗·교차표 ────────────────────────────────────────────────────────
# 요약 표 재배열



# 복습
- 파일 내보내기 실습

In [133]:
import pandas as pd
import seaborn as sns

tips = sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [134]:
# smoker = No인 데이터를 추출
tips.loc[tips['smoker'] == 'No', :].reset_index(drop=True).head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [135]:
# size >=3이면서, tip >= 3인 데이터를 추출
# tips.loc[(조건문1) & (조건문2), :]
tips.loc[(tips['size'] >= 3) & (tips['tip'] >= 3 ), :].reset_index(drop=True).head()

,total_bill,tip,sex,smoker,day,time,size
0,21.01,3.50,Male,No,Sun,Dinner,3
1,24.59,3.61,Female,No,Sun,Dinner,4
2,25.29,4.71,Male,No,Sun,Dinner,4
3,26.88,3.12,Male,No,Sun,Dinner,4
4,35.26,5.00,Female,No,Sun,Dinner,4


In [136]:
# 요일별 total_bill의 평균 금액
tips.groupby('day', observed=False)['total_bill'].mean()

,total_bill
day,
Thur,17.682742
Fri,17.151579
Sat,20.441379
Sun,21.410000


# 파일 내보내기
- csv로 내보내기
- excel로 내보내기

# CSV로 내보내기
- to_csv() 활용하기

In [137]:
result = tips.loc[(tips['size'] >= 3) & (tips['tip'] >= 3 ), :].reset_index(drop=True).copy()
result.to_csv("result.csv", index=False)

In [138]:
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/4주차/02_pandas와데이터시각화/output/'
result.to_csv(DATA_PATH + 'result.csv', index=False)

In [139]:
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/4주차/02_pandas와데이터시각화/output/'
result.to_excel(DATA_PATH + 'result.xlsx', sheet_name = 'tips')